In [ ]:
from ctrader_open_api import Client, Protobuf, TcpProtocol, Auth, EndPoints
from ctrader_open_api.messages.OpenApiCommonMessages_pb2 import *
from ctrader_open_api.messages.OpenApiMessages_pb2 import *
from ctrader_open_api.messages.OpenApiModelMessages_pb2 import *
from twisted.internet import reactor
import json
import datetime
import calendar
import keyring

In [ ]:
credentialsFile = open("credentials-dev.json")
credentials = json.load(credentialsFile)
credentials['Secret'] = keyring.get_password("ctrader", credentials['ClientId'])

In [ ]:
host = EndPoints.PROTOBUF_LIVE_HOST if credentials["HostType"].lower() == "live" else EndPoints.PROTOBUF_DEMO_HOST
client = Client(host, EndPoints.PROTOBUF_PORT, TcpProtocol)

In [ ]:
symbolName = "US500"

In [ ]:
dailyBars = []

In [ ]:
def transformTrendbar(trendbar):
    openTime = datetime.datetime.fromtimestamp(trendbar.utcTimestampInMinutes * 60, datetime.timezone.utc)
    openPrice = (trendbar.low + trendbar.deltaOpen) / 100000.0
    highPrice = (trendbar.low + trendbar.deltaHigh) / 100000.0
    lowPrice = trendbar.low / 100000.0
    closePrice = (trendbar.low + trendbar.deltaClose) / 100000.0
    return [openTime, openPrice, highPrice, lowPrice, closePrice, trendbar.volume]

In [ ]:
def trendbarsResponseCallback(result):
    print("\nTrendbars received")
    trendbars = Protobuf.extract(result)
    barsData = list(map(transformTrendbar, trendbars.trendbar))
    global dailyBars
    dailyBars.clear()
    dailyBars.extend(barsData)
    print("\ndailyBars length:", len(dailyBars))
    print("\Stopping reactor...")
    reactor.stop()

def symbolsResponseCallback(result):
    print("\nSymbols received")
    symbols = Protobuf.extract(result)
    global symbolName
    symbolsFilterResult = list(filter(lambda symbol: symbol.symbolName == symbolName, symbols.symbol))
    if len(symbolsFilterResult) == 0:
        raise Exception(f"There is symbol that matches to your defined symbol name: {symbolName}")
    elif len(symbolsFilterResult) > 1:
        raise Exception(f"More than one symbol matched with your defined symbol name: {symbolName}, match result: {symbolsFilterResult}")
    symbol = symbolsFilterResult[0]

    # Fetch multiple chunks
    num_chunks = 52*10
    weeks_per_chunk = 1
    now = datetime.datetime.utcnow()
    requests = []
    for i in range(num_chunks):
        to_time = now - datetime.timedelta(weeks=weeks_per_chunk * i)
        from_time = to_time - datetime.timedelta(weeks=weeks_per_chunk)
        request = ProtoOAGetTrendbarsReq()
        request.symbolId = symbol.symbolId
        request.ctidTraderAccountId = credentials["AccountId"]
        request.period = ProtoOATrendbarPeriod.M1
        request.fromTimestamp = int(calendar.timegm(from_time.utctimetuple())) * 1000
        request.toTimestamp = int(calendar.timegm(to_time.utctimetuple())) * 1000
        requests.append(request)

    # Helper to chain requests
    def fetch_next(index):
        if index >= len(requests):
            print("\nAll chunks fetched")
            reactor.stop()
            return
        deferred = client.send(requests[index])
        def on_success(result):
            trendbars = Protobuf.extract(result)
            barsData = list(map(transformTrendbar, trendbars.trendbar))
            global dailyBars
            dailyBars.extend(barsData)
            print(f"\nFetched chunk {index+1}/{len(requests)}, bars: {len(barsData)}")
            fetch_next(index + 1)
        deferred.addCallbacks(on_success, onError)

    # Start fetching
    global dailyBars
    dailyBars.clear()
    fetch_next(0)
    
def accountAuthResponseCallback(result):
    print("\nAccount authenticated")
    request = ProtoOASymbolsListReq()
    request.ctidTraderAccountId = credentials["AccountId"]
    request.includeArchivedSymbols = False
    deferred = client.send(request)
    deferred.addCallbacks(symbolsResponseCallback, onError)
    
def applicationAuthResponseCallback(result):
    print("\nApplication authenticated")
    request = ProtoOAAccountAuthReq()
    request.ctidTraderAccountId = credentials["AccountId"]
    request.accessToken = credentials["AccessToken"]
    deferred = client.send(request)
    deferred.addCallbacks(accountAuthResponseCallback, onError)

def onError(client, failure): # Call back for errors
    print("\nMessage Error: ", failure)

def disconnected(client, reason): # Callback for client disconnection
    print("\nDisconnected: ", reason)

def onMessageReceived(client, message): # Callback for receiving all messages
    if message.payloadType in [ProtoHeartbeatEvent().payloadType, ProtoOAAccountAuthRes().payloadType, ProtoOAApplicationAuthRes().payloadType, ProtoOASymbolsListRes().payloadType, ProtoOAGetTrendbarsRes().payloadType]:
        return
    print("\nMessage received: \n", Protobuf.extract(message))
    
def connected(client): # Callback for client connection
    print("\nConnected")
    request = ProtoOAApplicationAuthReq()
    request.clientId = credentials["ClientId"]
    request.clientSecret = credentials["Secret"]
    deferred = client.send(request)
    deferred.addCallbacks(applicationAuthResponseCallback, onError)
    
# Setting optional client callbacks
client.setConnectedCallback(connected)
client.setDisconnectedCallback(disconnected)
client.setMessageReceivedCallback(onMessageReceived)

In [ ]:
# Starting the client service
client.startService()

# Run Twisted reactor, we imported it earlier
reactor.run()

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.DataFrame(np.array(dailyBars),
                   columns=['Time', 'Open', 'High', 'Low', 'Close', 'Volume']).drop_duplicates().reset_index(drop=True)
df["Open"] = pd.to_numeric(df["Open"])
df["High"] = pd.to_numeric(df["High"])
df["Low"] = pd.to_numeric(df["Low"])
df["Close"] = pd.to_numeric(df["Close"])
df["Volume"] = pd.to_numeric(df["Volume"])

In [ ]:
df['Time'].describe()

In [ ]:
df = df.sort_values('Time')
df = df.drop_duplicates().reset_index(drop=True)
df.to_csv(f'../../data/{symbolName}_1minute.csv', index=False)

In [ ]:
df